# Dataset Assembly and Split

Joins labels, sensing features, and COVID features into the final dataset. Creates two versioned datasets: pre-COVID (v1) and full (v2).

Join logic: INNER JOIN on (uid, year_week) across labels and sensing.
COVID features LEFT JOIN.
Pre-COVID splits contain no COVID columns.
Full splits contain all columns including COVID items and covid_period.

Split boundaries:
- pre_covid_train, 2017-W37 to 2019-W26 - Train v1
- pre_covid_val, 2019-W27 to 2019-W45 - Tune v1
- pre_covid_test, 2019-W46 to 2020-W11 - Evaluate v1
- inference_v1, 2020-W06 to 2021-W26 - Drift demo
- full_train, 2017-W37 to 2020-W39 - Train v2
- full_val, 2020-W40 to 2021-W06 - Tune v2
- full_test, 2021-W07 to 2021-W26 - Evaluate v2
- inference_v2, 2021-W27 to 2022-W23 - Recovery demo

Outputs:
  - data/processed/splits/pre_covid_train.csv
  - data/processed/splits/pre_covid_val.csv
  - data/processed/splits/pre_covid_test.csv
  - data/processed/splits/inference_v1.csv
  - data/processed/splits/full_train.csv
  - data/processed/splits/full_val.csv
  - data/processed/splits/full_test.csv
  - data/processed/splits/inference_v2.csv

In [24]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path("../../")))

LABEL_PATH   = Path("../../data/processed/labels/weekly_labels.csv")
SENSING_PATH = Path("../../data/processed/features/sensing_weekly.csv")
COVID_PATH   = Path("../../data/processed/features/covid_weekly.csv")
OUTPUT_DIR   = Path("../../data/processed/splits")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRE_TRAIN_END  = "2019-W27"   
PRE_VAL_END    = "2019-W46"  
PRE_TEST_END   = "2020-W12"  

INF_V1_START   = "2020-W06"  
INF_V1_END     = "2021-W27"  

FULL_TRAIN_END = "2020-W40" 
FULL_VAL_END   = "2021-W07" 
FULL_TEST_END  = "2021-W27" 
INF_V2_START   = "2021-W27" 

COVID_BOUNDARY = "2020-W12"
COVID_COLS = [f"COVID-{i}" for i in range(1, 11) if i != 9] + ["covid_period"]

---
## Load All Three Sources

In [25]:
labels  = pd.read_csv(LABEL_PATH)
sensing = pd.read_csv(SENSING_PATH)
covid   = pd.read_csv(COVID_PATH)

# Rename feature_week -> year_week for join
labels = labels.rename(columns={"feature_week": "year_week"})

print("Loaded:")
print(f"  labels  : {len(labels):,} rows | {labels['uid'].nunique()} students")
print(f"  sensing : {len(sensing):,} rows | {sensing['uid'].nunique()} students")
print(f"  covid   : {len(covid):,} rows | {covid['uid'].nunique()} students")
print(f"\nSensing columns: {sorted(sensing.columns.tolist())}")
print(f"\nCOVID columns: {sorted(covid.columns.tolist())}")


Loaded:
  labels  : 20,193 rows | 216 students
  sensing : 24,775 rows | 215 students
  covid   : 28,925 rows | 216 students

Sensing columns: ['act_in_vehicle_ep_0_mean', 'act_in_vehicle_ep_0_std', 'act_on_bike_ep_0_mean', 'act_on_bike_ep_0_std', 'act_still_ep_0_mean', 'act_still_ep_0_std', 'is_ios', 'loc_self_dorm_dur_mean', 'loc_self_dorm_dur_std', 'loc_social_dur_mean', 'loc_social_dur_std', 'loc_study_dur_mean', 'loc_study_dur_std', 'other_playing_duration_ep_0_mean', 'other_playing_duration_ep_0_std', 'sleep_duration_mean', 'sleep_duration_std', 'uid', 'unlock_duration_ep_0_mean', 'unlock_duration_ep_0_std', 'unlock_num_ep_0_mean', 'unlock_num_ep_0_std', 'year_week']

COVID columns: ['COVID-1', 'COVID-10', 'COVID-2', 'COVID-3', 'COVID-4', 'COVID-5', 'COVID-6', 'COVID-7', 'COVID-8', 'covid_period', 'uid', 'year_week']


---
## Inner Join: Labels x Sensing

(student, week) pairs present in both tables.

In [26]:
assembled = labels.merge(
    sensing,
    on=["uid", "year_week"],
    how="inner"
)

# Left join COVID
assembled = assembled.merge(
    covid,
    on=["uid", "year_week"],
    how="left"
)

# Drop label_week 
assembled = assembled.drop(columns=["label_week"], errors="ignore")

print(f"Assembled dataset:")
print(f"  Rows     : {len(assembled):,}")
print(f"  Students : {assembled['uid'].nunique()}")
print(f"  Columns  : {assembled.shape[1]}")
print(f"  Range    : {assembled['year_week'].min()} to {assembled['year_week'].max()}")
print(f"\nColumn list:")
for col in assembled.columns:
    print(f"  {col}")


Assembled dataset:
  Rows     : 16,043
  Students : 212
  Columns  : 35
  Range    : 2017-W37 to 2022-W23

Column list:
  uid
  year_week
  label_composite_score
  n_surveys_in_week
  sleep_duration_mean
  sleep_duration_std
  unlock_num_ep_0_mean
  unlock_num_ep_0_std
  unlock_duration_ep_0_mean
  unlock_duration_ep_0_std
  act_still_ep_0_mean
  act_still_ep_0_std
  act_in_vehicle_ep_0_mean
  act_in_vehicle_ep_0_std
  act_on_bike_ep_0_mean
  act_on_bike_ep_0_std
  loc_self_dorm_dur_mean
  loc_self_dorm_dur_std
  loc_social_dur_mean
  loc_social_dur_std
  loc_study_dur_mean
  loc_study_dur_std
  other_playing_duration_ep_0_mean
  other_playing_duration_ep_0_std
  is_ios
  COVID-1
  COVID-2
  COVID-3
  COVID-4
  COVID-5
  COVID-6
  COVID-7
  COVID-8
  COVID-10
  covid_period


---
## Student Count Trace

Explains every student drop from 216 to the final assembled population. Completes the investigation started in phase1_data_understanding/01_dataset_inspection

In [27]:
uids_labels   = set(labels["uid"].unique())
uids_sensing  = set(sensing["uid"].unique())
uids_assembled = set(assembled["uid"].unique())

in_labels_not_sensing  = uids_labels - uids_sensing
in_sensing_not_labels  = uids_sensing - uids_labels
in_both_not_assembled  = (uids_labels & uids_sensing) - uids_assembled

print("Student count at each stage:")
print(f"  After EXCLUDE_UIDS (Phase 1)         : 216")
print(f"  In label table (weekly_labels.csv)   : {len(uids_labels)}")
print(f"  In sensing table (sensing_weekly.csv): {len(uids_sensing)}")
print(f"  In assembled dataset                 : {len(uids_assembled)}")

print(f"\nIn labels but NOT in sensing: {len(in_labels_not_sensing)}")
for uid in sorted(in_labels_not_sensing):
    n = (labels["uid"] == uid).sum()
    print(f"  {uid}")
    print(f"    {n} label rows - no quality sensing days (max quality_activity < 20h)")

print(f"\nIn sensing but NOT in labels: {len(in_sensing_not_labels)}")
for uid in sorted(in_sensing_not_labels):
    n = (sensing["uid"] == uid).sum()
    print(f"  {uid}")
    print(f"    {n} sensing weeks - no EMA label rows")

print(f"\nIn both but lost at join: {len(in_both_not_assembled)}")
for uid in sorted(in_both_not_assembled):
    lw = set(labels[labels["uid"] == uid]["year_week"])
    sw = set(sensing[sensing["uid"] == uid]["year_week"])
    overlap = lw & sw
    print(f"  {uid}")
    print(f"    {len(lw)} label weeks, {len(sw)} sensing weeks, "
          f"{len(overlap)} overlapping - "
          f"{'no overlapping weeks' if not overlap else 'check manually'}")

print(f"\nFinal assembled population: {len(uids_assembled)} students")

Student count at each stage:
  After EXCLUDE_UIDS (Phase 1)         : 216
  In label table (weekly_labels.csv)   : 216
  In sensing table (sensing_weekly.csv): 215
  In assembled dataset                 : 212

In labels but NOT in sensing: 1
  9ffbe25e279de21de86acd54b6fa60d0
    5 label rows - no quality sensing days (max quality_activity < 20h)

In sensing but NOT in labels: 0

In both but lost at join: 3
  1badfae62cc1b76787d4f8beb68737bf
    7 label weeks, 13 sensing weeks, 0 overlapping - no overlapping weeks
  1c12f39481a6c09da17bd4e6bebed9a2
    20 label weeks, 1 sensing weeks, 0 overlapping - no overlapping weeks
  c0b0998fe60a905081764378d1102494
    5 label weeks, 7 sensing weeks, 0 overlapping - no overlapping weeks

Final assembled population: 212 students


---
## Pre-COVID Splits (v1)

Three splits for v1 development without COVID columns.

- pre_covid_train : 2017-W37 to 2019-W26 - (~70% of pre-COVID data)
- pre_covid_val   : 2019-W27 to 2019-W45 - (~15%)
- pre_covid_test  : 2019-W46 to 2020-W11 - (~14%, bounded by COVID at W12)

In [28]:
pre_covid = assembled[assembled["year_week"] < COVID_BOUNDARY].copy()
pre_covid = pre_covid.drop(columns=COVID_COLS, errors="ignore")

pre_train = pre_covid[pre_covid["year_week"] < PRE_TRAIN_END].copy()
pre_val   = pre_covid[
    (pre_covid["year_week"] >= PRE_TRAIN_END) &
    (pre_covid["year_week"] < PRE_VAL_END)
].copy()
pre_test  = pre_covid[
    (pre_covid["year_week"] >= PRE_VAL_END) &
    (pre_covid["year_week"] < PRE_TEST_END)
].copy()

print("Pre-COVID splits (v1):")
print(f"  pre_covid_train : {len(pre_train):>6,} rows | "
      f"{pre_train['uid'].nunique()} students | "
      f"{pre_train['year_week'].min()} to {pre_train['year_week'].max()}")
print(f"  pre_covid_val   : {len(pre_val):>6,} rows | "
      f"{pre_val['uid'].nunique()} students | "
      f"{pre_val['year_week'].min()} to {pre_val['year_week'].max()}")
print(f"  pre_covid_test  : {len(pre_test):>6,} rows | "
      f"{pre_test['uid'].nunique()} students | "
      f"{pre_test['year_week'].min()} to {pre_test['year_week'].max()}")

total_pre = len(pre_train) + len(pre_val) + len(pre_test)
print(f"\n  Ratio — train: {len(pre_train)/total_pre*100:.0f}% | "
      f"val: {len(pre_val)/total_pre*100:.0f}% | "
      f"test: {len(pre_test)/total_pre*100:.0f}%")
print(f"\n  Columns: {pre_train.shape[1]} (no COVID columns)")

Pre-COVID splits (v1):
  pre_covid_train :  6,940 rows | 206 students | 2017-W37 to 2019-W26
  pre_covid_val   :  1,610 rows | 163 students | 2019-W27 to 2019-W45
  pre_covid_test  :  1,381 rows | 156 students | 2019-W46 to 2020-W11

  Ratio — train: 70% | val: 16% | test: 14%

  Columns: 25 (no COVID columns)


---
## Inference v1

The data replayed through v1 in Phase 5. 
Starts at W06 2020 (6 weeks before COVID) to show baseline performance.
Ends at W26 2021 when v2 takes over inference.
Sensing columns only.

In [29]:
inference_v1 = assembled[
    (assembled["year_week"] >= INF_V1_START) &
    (assembled["year_week"] < INF_V1_END)
].copy()
inference_v1 = inference_v1.drop(columns=COVID_COLS, errors="ignore")

pre_covid_weeks = inference_v1[
    inference_v1["year_week"] < COVID_BOUNDARY
]["year_week"].nunique()
covid_weeks = inference_v1[
    inference_v1["year_week"] >= COVID_BOUNDARY
]["year_week"].nunique()

print(f"inference_v1:")
print(f"  Rows             : {len(inference_v1):,}")
print(f"  Students         : {inference_v1['uid'].nunique()}")
print(f"  Range            : {inference_v1['year_week'].min()} "
      f"to {inference_v1['year_week'].max()}")
print(f"  Pre-COVID weeks  : {pre_covid_weeks} (baseline performance)")
print(f"  COVID weeks      : {covid_weeks} (drift period)")
print(f"  Columns          : {inference_v1.shape[1]}")

inference_v1:
  Rows             : 5,424
  Students         : 172
  Range            : 2020-W06 to 2021-W26
  Pre-COVID weeks  : 6 (baseline performance)
  COVID weeks      : 68 (drift period)
  Columns          : 25


---
## Full Dataset Splits (v2)

Three splits for v2 development plus inference data.
COVID columns present with real values from 2020-W12 onward.
Pre-COVID rows have covid_period=0 and NaN for COVID items.

- full_train   : 2017-W37 to 2020-W39 - (everything at retraining time, ~80%)
- full_val     : 2020-W40 to 2021-W06 - (post-COVID, ~10%)
- full_test    : 2021-W07 to 2021-W26 - (post-COVID, ~10%)
- inference_v2 : 2021-W27 to 2022-W23 - (Phase 5 recovery demo)

In [30]:
full_train = assembled[assembled["year_week"] < FULL_TRAIN_END].copy()
full_val   = assembled[
    (assembled["year_week"] >= FULL_TRAIN_END) &
    (assembled["year_week"] < FULL_VAL_END)
].copy()
full_test  = assembled[
    (assembled["year_week"] >= FULL_VAL_END) &
    (assembled["year_week"] < FULL_TEST_END)
].copy()
inference_v2 = assembled[assembled["year_week"] >= INF_V2_START].copy()

dev_total = len(full_train) + len(full_val) + len(full_test)

print("Full dataset splits (v2):")
print(f"  full_train   : {len(full_train):>6,} rows | "
      f"{full_train['uid'].nunique()} students | "
      f"{full_train['year_week'].min()} to {full_train['year_week'].max()}")
print(f"  full_val     : {len(full_val):>6,} rows | "
      f"{full_val['uid'].nunique()} students | "
      f"{full_val['year_week'].min()} to {full_val['year_week'].max()}")
print(f"  full_test    : {len(full_test):>6,} rows | "
      f"{full_test['uid'].nunique()} students | "
      f"{full_test['year_week'].min()} to {full_test['year_week'].max()}")
print(f"  inference_v2 : {len(inference_v2):>6,} rows | "
      f"{inference_v2['uid'].nunique()} students | "
      f"{inference_v2['year_week'].min()} to {inference_v2['year_week'].max()}")

print(f"\n  Model dev ratio (train+val+test={dev_total:,}):")
print(f"    train: {len(full_train)/dev_total*100:.0f}% | "
      f"val: {len(full_val)/dev_total*100:.0f}% | "
      f"test: {len(full_test)/dev_total*100:.0f}%")

covid_nan_train = full_train["COVID-1"].isna().mean() * 100
covid_nan_test  = full_test["COVID-1"].isna().mean() * 100
print(f"\n  COVID-1 NaN rate in full_train : {covid_nan_train:.1f}%")
print(f"  COVID-1 NaN rate in full_test  : {covid_nan_test:.1f}%")
print(f"  (full_test is post-COVID so COVID items should be mostly populated)")

Full dataset splits (v2):
  full_train   : 12,162 rows | 212 students | 2017-W37 to 2020-W39
  full_val     :  1,374 rows | 149 students | 2020-W40 to 2021-W06
  full_test    :  1,223 rows | 127 students | 2021-W07 to 2021-W26
  inference_v2 :  1,284 rows | 73 students | 2021-W27 to 2022-W23

  Model dev ratio (train+val+test=14,759):
    train: 82% | val: 9% | test: 8%

  COVID-1 NaN rate in full_train : 87.8%
  COVID-1 NaN rate in full_test  : 54.7%
  (full_test is post-COVID so COVID items should be mostly populated)


---
## Verification

In [31]:
print("Temporal overlap checks:")

# Pre-COVID splits
if pre_train["year_week"].max() >= pre_val["year_week"].min():
    print("  WARNING: pre_covid_train/val overlap")
elif pre_val["year_week"].max() >= pre_test["year_week"].min():
    print("  WARNING: pre_covid_val/test overlap")
else:
    print("  OK: pre-COVID train/val/test have no temporal overlap")

# Full splits
if full_train["year_week"].max() >= full_val["year_week"].min():
    print("  WARNING: full_train/val overlap")
elif full_val["year_week"].max() >= full_test["year_week"].min():
    print("  WARNING: full_val/test overlap")
elif full_test["year_week"].max() >= inference_v2["year_week"].min():
    print("  WARNING: full_test/inference_v2 overlap")
else:
    print("  OK: full train/val/test/inference have no temporal overlap")

# COVID columns absent from pre-COVID splits
covid_in_pre = [c for c in COVID_COLS if c in pre_train.columns]
if covid_in_pre:
    print(f"  WARNING: COVID columns found in pre-COVID splits: {covid_in_pre}")
else:
    print("  OK: no COVID columns in pre-COVID splits")

# Labels are in expected range
for name, df in [("pre_train", pre_train), ("full_test", full_test)]:
    lo = df["label_composite_score"].min()
    hi = df["label_composite_score"].max()
    mn = df["label_composite_score"].mean()
    if lo < 0 or hi > 100:
        print(f"  WARNING: {name} label out of [0,100] range: [{lo:.1f}, {hi:.1f}]")
    else:
        print(f"  OK: {name} labels in [0,100] — mean={mn:.1f}, "
              f"min={lo:.1f}, max={hi:.1f}")

# pre_covid_test ends before COVID boundary
if pre_test["year_week"].max() >= COVID_BOUNDARY:
    print(f"  WARNING: pre_covid_test extends past COVID boundary {COVID_BOUNDARY}")
else:
    print(f"  OK: pre_covid_test ends before COVID boundary {COVID_BOUNDARY}")


Temporal overlap checks:
  OK: pre-COVID train/val/test have no temporal overlap
  OK: full train/val/test/inference have no temporal overlap
  OK: no COVID columns in pre-COVID splits
  OK: pre_train labels in [0,100] — mean=64.8, min=0.0, max=100.0
  OK: full_test labels in [0,100] — mean=60.2, min=0.0, max=99.2
  OK: pre_covid_test ends before COVID boundary 2020-W12


---
## Summary Statistics

In [32]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

splits = {
    "pre_covid_train": pre_train,
    "pre_covid_val":   pre_val,
    "pre_covid_test":  pre_test,
    "inference_v1":    inference_v1,
    "full_train":      full_train,
    "full_val":        full_val,
    "full_test":       full_test,
    "inference_v2":    inference_v2,
}

for name, df in splits.items():
    print(f"\n  {name}")
    print(f"    Rows     : {len(df):,}")
    print(f"    Students : {df['uid'].nunique()}")
    print(f"    Columns  : {df.shape[1]}")
    print(f"    Period   : {df['year_week'].min()} to {df['year_week'].max()}")
    if "label_composite_score" in df.columns:
        print(f"    Label    : mean={df['label_composite_score'].mean():.1f}, "
              f"std={df['label_composite_score'].std():.1f}")

FINAL DATASET SUMMARY

  pre_covid_train
    Rows     : 6,940
    Students : 206
    Columns  : 25
    Period   : 2017-W37 to 2019-W26
    Label    : mean=64.8, std=15.8

  pre_covid_val
    Rows     : 1,610
    Students : 163
    Columns  : 25
    Period   : 2019-W27 to 2019-W45
    Label    : mean=64.1, std=16.2

  pre_covid_test
    Rows     : 1,381
    Students : 156
    Columns  : 25
    Period   : 2019-W46 to 2020-W11
    Label    : mean=63.6, std=16.4

  inference_v1
    Rows     : 5,424
    Students : 172
    Columns  : 25
    Period   : 2020-W06 to 2021-W26
    Label    : mean=60.5, std=16.2

  full_train
    Rows     : 12,162
    Students : 212
    Columns  : 35
    Period   : 2017-W37 to 2020-W39
    Label    : mean=63.8, std=16.1

  full_val
    Rows     : 1,374
    Students : 149
    Columns  : 35
    Period   : 2020-W40 to 2021-W06
    Label    : mean=60.1, std=15.6

  full_test
    Rows     : 1,223
    Students : 127
    Columns  : 35
    Period   : 2021-W07 to 2021-W26


---
## Save

In [33]:
for name, df in splits.items():
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    size_kb = path.stat().st_size / 1024
    print(f"  {name}.csv : {len(df):,} rows, "
          f"{df.shape[1]} cols, {size_kb:.0f} KB")

print(f"  v1: pre_covid_train ({len(pre_train):,} examples, "
      f"{pre_train.shape[1]} cols)")
print(f"  v2: full_train ({len(full_train):,} examples, "
      f"{full_train.shape[1]} cols)")

  pre_covid_train.csv : 6,940 rows, 25 cols, 2346 KB
  pre_covid_val.csv : 1,610 rows, 25 cols, 532 KB
  pre_covid_test.csv : 1,381 rows, 25 cols, 455 KB
  inference_v1.csv : 5,424 rows, 25 cols, 1604 KB
  full_train.csv : 12,162 rows, 35 cols, 4137 KB
  full_val.csv : 1,374 rows, 35 cols, 432 KB
  full_test.csv : 1,223 rows, 35 cols, 398 KB
  inference_v2.csv : 1,284 rows, 35 cols, 447 KB
  v1: pre_covid_train (6,940 examples, 25 cols)
  v2: full_train (12,162 examples, 35 cols)
